# KAIST AI College RAG 평가 노트북

이 노트북은 `src/rag/rag_pipeline.py` 기준으로 예상 질문을 실행하고, 단순 답변 길이가 아니라 **근거/라우팅/검색 상태**까지 함께 평가합니다.

핵심 평가 기준:

- `expected='answer'`인데 `source_count=0`이면 실패로 봅니다.
- 답변이 길어도 `"제공된 자료에서 확인할 수 없습니다"` 계열이면 PASS가 아니라 `FAIL_DATA_MISSING` 또는 `PARTIAL_NO_DIRECT_EVIDENCE`로 분리합니다.
- `route`, `intent`, `department_code`, `department_codes`, `sql_row_count`, `vector_result_count`, `warnings`를 함께 저장합니다.
- 데이터 부족인지, RAG 라우팅 문제인지, SQL 문제인지, Vector 검색 문제인지 `issue_type`으로 분류합니다.

In [1]:
# ============================================================
# 0. 기본 설정
# ============================================================

from __future__ import annotations

import json
import os
import sys
import time
import traceback
from dataclasses import asdict
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd

try:
    from dotenv import load_dotenv
except ModuleNotFoundError:
    def load_dotenv(*args, **kwargs):
        return False

# notebook 위치가 프로젝트 루트/notebooks 안에 있다고 가정
NOTEBOOK_DIR = Path.cwd()

if (NOTEBOOK_DIR / "src").exists():
    PROJECT_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "src").exists():
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    # 필요하면 직접 수정
    PROJECT_ROOT = Path(r"C:\Users\Playdata\workspace\SKN28-third-2TEAM")

PROJECT_ROOT = PROJECT_ROOT.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "chatbot_eval_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("OPENAI_API_KEY exists:", bool(os.getenv("OPENAI_API_KEY")))

PROJECT_ROOT: C:\Users\Playdata\workspace\SKN28-third-2TEAM
OUTPUT_DIR: C:\Users\Playdata\workspace\SKN28-third-2TEAM\notebooks\chatbot_eval_outputs
OPENAI_API_KEY exists: True


In [2]:
# ============================================================
# 1. Pipeline 로드
# ============================================================

from src.rag.rag_pipeline import create_default_pipeline

pipeline = create_default_pipeline(
    include_sql=True,
    include_debug_context=True,
    preload_vector_retriever=True,
    preload_answer_generator=True,
)

startup_warnings = getattr(pipeline, "_startup_warnings", [])
print("Pipeline loaded.")
print("startup_warnings:", startup_warnings)

Pipeline loaded.
startup_warnings: []


In [3]:
# ============================================================
# 2. 예상 질문 100개
# expected:
# - answer: 현재 데이터/구조상 답변이 나와야 하는 질문
# - reject: 범위 밖이라 거절해야 하는 질문
# - data_missing: 자료 부족을 명확히 말해야 하는 질문
# ============================================================

TEST_CASES = [
    # A. 학과 소개 / 개요
    {"id": 1, "category": "department_overview", "question": "AI컴퓨팅학과는 어떤 학과야?", "expected": "answer"},
    {"id": 2, "category": "department_overview", "question": "AI시스템학과는 어떤 학과야?", "expected": "answer"},
    {"id": 3, "category": "department_overview", "question": "AX학과는 어떤 학과야?", "expected": "answer"},
    {"id": 4, "category": "department_overview", "question": "AI미래학과는 어떤 학과야?", "expected": "answer"},
    {"id": 5, "category": "department_overview", "question": "AI미래학과는 어떤 인재를 양성하려고 해?", "expected": "answer"},
    {"id": 6, "category": "department_overview", "question": "AI컴퓨팅학과의 교육목표를 알려줘.", "expected": "answer"},
    {"id": 7, "category": "department_overview", "question": "AI시스템학과의 특징을 설명해줘.", "expected": "answer"},
    {"id": 8, "category": "department_overview", "question": "AX학과의 목표를 알려줘.", "expected": "answer"},
    {"id": 9, "category": "department_overview", "question": "AI대학에는 어떤 학과들이 있어?", "expected": "answer"},
    {"id": 10, "category": "department_overview", "question": "KAIST AI대학 학과들을 간단히 소개해줘.", "expected": "answer"},

    # B. 비교
    {"id": 11, "category": "comparison", "question": "AI컴퓨팅학과와 AX학과를 비교해줘.", "expected": "answer"},
    {"id": 12, "category": "comparison", "question": "AI시스템학과와 AI미래학과 차이를 알려줘.", "expected": "answer"},
    {"id": 13, "category": "comparison", "question": "AI컴퓨팅학과와 AI시스템학과를 비교해줘.", "expected": "answer"},
    {"id": 14, "category": "comparison", "question": "AX학과와 AI미래학과의 차이점은 뭐야?", "expected": "answer"},
    {"id": 15, "category": "comparison", "question": "AI대학 학과들을 표 형식으로 비교해줘.", "expected": "answer"},
    {"id": 16, "category": "comparison", "question": "AI대학 학과별 특징을 비교해서 알려줘.", "expected": "answer"},
    {"id": 17, "category": "comparison", "question": "AI컴퓨팅학과와 AX학과의 공통점과 차이점을 알려줘.", "expected": "answer"},
    {"id": 18, "category": "comparison", "question": "AI시스템학과와 AX학과 중 산업 적용과 더 가까운 곳은 어디야?", "expected": "answer"},
    {"id": 19, "category": "comparison", "question": "AI대학 학과별 교육 방향을 비교해줘.", "expected": "answer"},
    {"id": 20, "category": "comparison", "question": "AI대학 학과 비교 내용을 출처 기반으로 설명해줘.", "expected": "answer"},

    # C. 추천
    {"id": 21, "category": "recommendation", "question": "나는 AI 서비스 적용에 관심이 있는데 어떤 학과가 맞아?", "expected": "answer"},
    {"id": 22, "category": "recommendation", "question": "산업 현장에 AI를 적용하는 분야에 관심이 있어. 어떤 학과가 적합해?", "expected": "answer"},
    {"id": 23, "category": "recommendation", "question": "AI 반도체나 하드웨어보다 소프트웨어 쪽에 관심이 있어. 어떤 학과를 보면 좋을까?", "expected": "answer"},
    {"id": 24, "category": "recommendation", "question": "미래 AI 기술과 사회 변화에 관심이 있는데 어떤 학과가 맞아?", "expected": "answer"},
    {"id": 25, "category": "recommendation", "question": "AI 서비스 기획과 개발을 같이 하고 싶다면 어떤 학과 정보를 먼저 봐야 해?", "expected": "answer"},
    {"id": 26, "category": "recommendation", "question": "대규모 AI 시스템 개발에 관심이 있으면 어떤 학과를 추천해?", "expected": "answer"},
    {"id": 27, "category": "recommendation", "question": "AI 인프라와 시스템 구조에 관심이 있어. 어떤 학과가 좋아?", "expected": "answer"},
    {"id": 28, "category": "recommendation", "question": "AI를 기업 업무에 적용하는 쪽을 배우고 싶어. 어느 학과가 맞아?", "expected": "answer"},
    {"id": 29, "category": "recommendation", "question": "AI 정책이나 사회 변화에 관심 있는 학생은 어떤 학과를 봐야 해?", "expected": "answer"},
    {"id": 30, "category": "recommendation", "question": "컴퓨팅 기반 AI 개발에 관심 있으면 어떤 학과가 적합해?", "expected": "answer"},

    # D. 교과목 / 교육과정
    {"id": 31, "category": "course", "question": "AI컴퓨팅학과의 교육과정을 알려줘.", "expected": "answer"},
    {"id": 32, "category": "course", "question": "AI시스템학과의 교육과정을 알려줘.", "expected": "data_missing"},
    {"id": 33, "category": "course", "question": "AX학과의 교육과정을 알려줘.", "expected": "answer"},
    {"id": 34, "category": "course", "question": "AI미래학과의 교육과정을 알려줘.", "expected": "answer"},
    {"id": 35, "category": "course", "question": "AI대학에서 제공하는 교과목 정보를 정리해줘.", "expected": "answer"},
    {"id": 36, "category": "course", "question": "AI컴퓨팅학과에서 배울 수 있는 과목을 알려줘.", "expected": "answer"},
    {"id": 37, "category": "course", "question": "AI시스템학과에서 배울 수 있는 과목을 알려줘.", "expected": "data_missing"},
    {"id": 38, "category": "course", "question": "AX학과 과목 목록을 알려줘.", "expected": "answer"},
    {"id": 39, "category": "course", "question": "AI미래학과 과목 목록을 알려줘.", "expected": "answer"},
    {"id": 40, "category": "course", "question": "AI대학 학과별 주요 교과목을 비교해줘.", "expected": "answer"},

    # E. 교수진
    {"id": 41, "category": "person", "question": "AI컴퓨팅학과 교수진을 알려줘.", "expected": "answer"},
    {"id": 42, "category": "person", "question": "AI시스템학과 교수진을 알려줘.", "expected": "answer"},
    {"id": 43, "category": "person", "question": "AX학과 교수진을 알려줘.", "expected": "answer"},
    {"id": 44, "category": "person", "question": "AI미래학과 교수진을 알려줘.", "expected": "answer"},
    {"id": 45, "category": "person", "question": "AI대학 교수진 목록을 정리해줘.", "expected": "answer"},
    {"id": 46, "category": "person_research_area", "question": "AI대학 교수들의 연구 분야를 알려줘.", "expected": "data_missing"},
    {"id": 47, "category": "person_research_area", "question": "AI컴퓨팅 관련 연구를 하는 교수는 누구야?", "expected": "data_missing"},
    {"id": 48, "category": "person", "question": "AI미래학과 교수 이메일을 알려줘.", "expected": "answer"},
    {"id": 49, "category": "person_research_area", "question": "AX 관련 연구를 하는 교수진 정보가 있으면 알려줘.", "expected": "data_missing"},
    {"id": 50, "category": "person", "question": "AI대학 교수 홈페이지나 이메일 정보가 있으면 알려줘.", "expected": "answer"},

    # F. 졸업/수료/논문 요건
    {"id": 51, "category": "requirement", "question": "AI컴퓨팅학과의 졸업 요건이 문서에 나와 있어?", "expected": "data_missing"},
    {"id": 52, "category": "requirement", "question": "AI시스템학과의 졸업 요건을 알려줘.", "expected": "data_missing"},
    {"id": 53, "category": "requirement", "question": "AX학과의 졸업 요건을 알려줘.", "expected": "data_missing"},
    {"id": 54, "category": "requirement", "question": "AI미래학과의 졸업 요건을 알려줘.", "expected": "data_missing"},
    {"id": 55, "category": "requirement", "question": "AI대학 학과별 이수 요건을 비교해줘.", "expected": "data_missing"},
    {"id": 56, "category": "requirement", "question": "AI대학 석사 과정 이수 요건이 있으면 알려줘.", "expected": "data_missing"},
    {"id": 57, "category": "requirement", "question": "AI대학 박사 과정 이수 요건이 있으면 알려줘.", "expected": "data_missing"},
    {"id": 58, "category": "requirement", "question": "AI대학 졸업학점 기준이 나와 있어?", "expected": "data_missing"},
    {"id": 59, "category": "requirement", "question": "AI대학 논문 제출 요건이 문서에 나와 있으면 알려줘.", "expected": "data_missing"},
    {"id": 60, "category": "requirement", "question": "AI대학 수료 요건과 졸업 요건을 구분해서 설명해줘.", "expected": "data_missing"},

    # G. 입학
    {"id": 61, "category": "admission", "question": "AI컴퓨팅학과 석사 지원 자격을 알려줘.", "expected": "answer"},
    {"id": 62, "category": "admission", "question": "AI시스템학과 입학 정보를 알려줘.", "expected": "answer"},
    {"id": 63, "category": "admission", "question": "AX학과 지원 자격을 알려줘.", "expected": "answer"},
    {"id": 64, "category": "admission", "question": "AI미래학과 입학 정보를 알려줘.", "expected": "answer"},
    {"id": 65, "category": "admission", "question": "AI대학 학과별 입학 정보를 비교해줘.", "expected": "answer"},
    {"id": 66, "category": "admission", "question": "AI컴퓨팅학과 제출 서류가 나와 있어?", "expected": "answer"},
    {"id": 67, "category": "admission", "question": "AX학과 입학 일정 알려줘.", "expected": "answer"},
    {"id": 68, "category": "admission", "question": "AI미래학과 석사 지원 조건 알려줘.", "expected": "answer"},
    {"id": 69, "category": "admission", "question": "AI시스템학과 박사 과정 입학 정보 알려줘.", "expected": "answer"},
    {"id": 70, "category": "admission", "question": "AI대학 입학 관련 출처를 함께 알려줘.", "expected": "answer"},

    # H. 연락처/홈페이지/위치
    {"id": 71, "category": "contact_location", "question": "AI대학의 위치 정보가 있으면 알려줘.", "expected": "data_missing"},
    {"id": 72, "category": "contact", "question": "KAIST 학과사무실 전화번호 알려줘.", "expected": "answer"},
    {"id": 73, "category": "contact", "question": "AI컴퓨팅학과의 연락처가 문서에 나와 있어?", "expected": "answer"},
    {"id": 74, "category": "contact", "question": "AI시스템학과의 연락처가 문서에 나와 있어?", "expected": "answer"},
    {"id": 75, "category": "contact", "question": "AX학과의 연락처가 문서에 나와 있어?", "expected": "answer"},
    {"id": 76, "category": "contact", "question": "AI미래학과의 연락처가 문서에 나와 있어?", "expected": "answer"},
    {"id": 77, "category": "homepage", "question": "AI컴퓨팅학과 홈페이지 알려줘.", "expected": "answer"},
    {"id": 78, "category": "homepage", "question": "AX학과 사이트 알려줘.", "expected": "answer"},
    {"id": 79, "category": "contact_location", "question": "AI대학 행정실 정보가 문서에 나와 있으면 알려줘.", "expected": "data_missing"},
    {"id": 80, "category": "homepage", "question": "AI대학 학과별 홈페이지 URL을 정리해줘.", "expected": "answer"},

    # I. 출처 기반
    {"id": 81, "category": "source_based", "question": "AI컴퓨팅학과 소개를 출처와 함께 알려줘.", "expected": "answer"},
    {"id": 82, "category": "source_based", "question": "AI시스템학과 입학 정보를 출처와 함께 알려줘.", "expected": "answer"},
    {"id": 83, "category": "source_based", "question": "AX학과 설명을 근거 문서와 함께 알려줘.", "expected": "answer"},
    {"id": 84, "category": "source_based", "question": "AI미래학과 소개를 출처 기반으로 알려줘.", "expected": "answer"},
    {"id": 85, "category": "source_based", "question": "AI대학 학과별 교육과정 정보를 출처와 함께 알려줘.", "expected": "answer"},
    {"id": 86, "category": "source_based", "question": "AI대학 교수진 정보를 출처와 함께 알려줘.", "expected": "answer"},
    {"id": 87, "category": "source_based", "question": "AI대학 입학 정보를 문서 근거와 함께 요약해줘.", "expected": "answer"},
    {"id": 88, "category": "source_based_requirement", "question": "AI대학 졸업 요건을 문서 근거와 함께 설명해줘.", "expected": "data_missing"},
    {"id": 89, "category": "source_based", "question": "AI대학 학과 비교 내용을 출처 기반으로 설명해줘.", "expected": "answer"},
    {"id": 90, "category": "source_based", "question": "AI대학 학과별 홈페이지를 출처와 함께 알려줘.", "expected": "answer"},

    # J. 범위 밖 / 거절
    {"id": 91, "category": "reject", "question": "서울대학교 AI대학원 입학 정보를 알려줘.", "expected": "reject"},
    {"id": 92, "category": "reject", "question": "KAIST 전산학부 교수진을 알려줘.", "expected": "reject"},
    {"id": 93, "category": "reject", "question": "AI대학 학과별 경쟁률을 알려줘.", "expected": "reject"},
    {"id": 94, "category": "reject", "question": "AI대학 등록금 얼마야?", "expected": "reject"},
    {"id": 95, "category": "reject", "question": "AI대학 졸업 후 평균 연봉 알려줘.", "expected": "reject"},
    {"id": 96, "category": "reject", "question": "내가 AI컴퓨팅학과에 합격할 가능성이 얼마나 돼?", "expected": "reject"},
    {"id": 97, "category": "reject", "question": "오늘 대전 날씨 알려줘.", "expected": "reject"},
    {"id": 98, "category": "reject", "question": "파이썬으로 웹 크롤러 코드 짜줘.", "expected": "reject"},
    {"id": 99, "category": "reject", "question": "서울대학교 AI대학원과 KAIST AI대학을 교수진 기준으로 비교해줘.", "expected": "reject"},
    {"id": 100, "category": "reject", "question": "KAIST 기계공학과 교과목 알려줘.", "expected": "reject"},
]

len(TEST_CASES)

100

In [4]:
# ============================================================
# 3. 평가 유틸 함수
# ============================================================

NO_EVIDENCE_PHRASES = [
    "제공된 자료에서 확인할 수 없습니다",
    "직접 근거를 찾을 수 없습니다",
    "근거를 찾을 수 없습니다",
    "자료에서 확인되지 않습니다",
    "자료가 부족",
    "데이터가 부족",
    "현재 데이터에는",
    "현재 보유 데이터에는",
    "현재 수집 자료에는",
    "찾지 못했습니다",
    "검색 결과 없음",
]

REJECT_PHRASES = [
    "수집 범위",
    "범위 밖",
    "확인하기 어렵습니다",
    "공식 홈페이지",
    "입학처에서 확인",
    "변동성이 크",
    "질문으로 다시 입력",
    "현재 수집된 데이터",
]

def safe_json_dumps(value: Any) -> str:
    try:
        return json.dumps(value, ensure_ascii=False, default=str)
    except Exception:
        return str(value)


def get_attr_or_key(obj: Any, key: str, default: Any = None) -> Any:
    if obj is None:
        return default
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def extract_sources(result: Any) -> list[dict[str, Any]]:
    sources = get_attr_or_key(result, "sources", [])
    if sources is None:
        return []
    return list(sources)


def extract_sql_rows(result: Any) -> list[dict[str, Any]]:
    sql_result = get_attr_or_key(result, "sql_result")
    if sql_result is None:
        return []
    rows = get_attr_or_key(sql_result, "rows", [])
    return list(rows or [])


def extract_vector_result(result: Any) -> Any:
    return get_attr_or_key(result, "vector_result")


def extract_vector_count(result: Any) -> int:
    vector_result = extract_vector_result(result)
    if vector_result is None:
        return 0
    results = get_attr_or_key(vector_result, "results", [])
    if results is not None:
        return len(results)
    docs = get_attr_or_key(vector_result, "documents", [])
    return len(docs or [])


def extract_vector_status(result: Any) -> str | None:
    vector_result = extract_vector_result(result)
    if vector_result is None:
        return None
    return get_attr_or_key(vector_result, "status")


def extract_vector_used_fallback(result: Any) -> bool:
    vector_result = extract_vector_result(result)
    if vector_result is None:
        return False
    return bool(get_attr_or_key(vector_result, "used_fallback", False))


def extract_analysis(result: Any) -> Any:
    return get_attr_or_key(result, "analysis")


def extract_warnings(result: Any) -> list[str]:
    warnings = []
    result_warnings = get_attr_or_key(result, "warnings", []) or []
    warnings.extend([str(w) for w in result_warnings if w])

    vector_result = extract_vector_result(result)
    if vector_result is not None:
        vector_warnings = get_attr_or_key(vector_result, "warnings", []) or []
        warnings.extend([str(w) for w in vector_warnings if w])

    sql_result = get_attr_or_key(result, "sql_result")
    if sql_result is not None:
        sql_warnings = get_attr_or_key(sql_result, "warnings", []) or []
        warnings.extend([str(w) for w in sql_warnings if w])

    # 중복 제거
    deduped = []
    seen = set()
    for warning in warnings:
        if warning in seen:
            continue
        seen.add(warning)
        deduped.append(warning)
    return deduped


def contains_any(text: str, phrases: list[str]) -> bool:
    text = str(text or "")
    return any(phrase in text for phrase in phrases)


def classify_issue_type(
    expected: str,
    answer: str,
    route: str | None,
    intent: str | None,
    source_count: int,
    sql_row_count: int,
    vector_status: str | None,
    vector_result_count: int,
    used_fallback: bool,
    warnings: list[str],
) -> str:
    warning_text = " ".join(warnings)

    if expected == "reject":
        if contains_any(answer, REJECT_PHRASES) or route == "clarify":
            return "EXPECTED_REJECTION"
        return "REJECT_FAILED"

    if expected == "data_missing":
        if contains_any(answer, NO_EVIDENCE_PHRASES):
            return "EXPECTED_DATA_MISSING"
        return "DATA_MISSING_NOT_DETECTED"

    # expected == answer
    if route == "clarify":
        return "UNEXPECTED_CLARIFY"

    if "SQLTool이 설정되어 있지 않습니다" in warning_text or "sql_error" in warning_text:
        return "SQL_ERROR"

    if source_count == 0:
        return "NO_SOURCE"

    if contains_any(answer, NO_EVIDENCE_PHRASES):
        if sql_row_count == 0 and (vector_status in {None, "no_result", "skipped_sql_route"} or vector_result_count == 0):
            return "DATA_MISSING_OR_RETRIEVAL_FAIL"
        return "PARTIAL_NO_DIRECT_EVIDENCE"

    if used_fallback:
        return "ANSWER_WITH_FALLBACK"

    return "OK"


def basic_evaluate(row: dict[str, Any]) -> str:
    expected = row["expected"]
    answer = row["answer"]
    source_count = row["source_count"]
    route = row["route"]
    issue_type = row["issue_type"]

    if expected == "reject":
        if issue_type == "EXPECTED_REJECTION":
            return "PASS_REJECT"
        return "FAIL_SHOULD_REJECT"

    if expected == "data_missing":
        if issue_type == "EXPECTED_DATA_MISSING":
            return "PASS_DATA_MISSING"
        return "FAIL_DATA_MISSING_NOT_DETECTED"

    # expected == answer
    if issue_type == "OK":
        return "PASS_ANSWER"

    if issue_type == "ANSWER_WITH_FALLBACK":
        return "PARTIAL_FALLBACK"

    if issue_type == "PARTIAL_NO_DIRECT_EVIDENCE":
        return "PARTIAL_NO_DIRECT_EVIDENCE"

    if issue_type == "DATA_MISSING_OR_RETRIEVAL_FAIL":
        return "FAIL_DATA_MISSING_OR_RETRIEVAL"

    if issue_type == "NO_SOURCE":
        return "FAIL_NO_SOURCE"

    if issue_type == "UNEXPECTED_CLARIFY":
        return "FAIL_UNEXPECTED_CLARIFY"

    if issue_type == "SQL_ERROR":
        return "FAIL_SQL_ERROR"

    if len(str(answer).strip()) < 20:
        return "FAIL_WEAK_ANSWER"

    # 알 수 없는 문제는 실패로 두는 것이 안전
    return "FAIL_CHECK_MANUALLY"

In [5]:
# ============================================================
# 4. 단일 질문 실행 함수
# ============================================================

def run_one_case(case: dict[str, Any]) -> dict[str, Any]:
    qid = case["id"]
    question = case["question"]
    category = case["category"]
    expected = case["expected"]

    started_at = time.time()

    base = {
        "id": qid,
        "category": category,
        "question": question,
        "expected": expected,
    }

    try:
        result = pipeline.run(question)
        elapsed_sec = round(time.time() - started_at, 3)

        analysis = extract_analysis(result)
        answer = get_attr_or_key(result, "answer", "")

        route = get_attr_or_key(analysis, "route")
        intent = get_attr_or_key(analysis, "intent")
        department_code = get_attr_or_key(analysis, "department_code")
        department_codes = get_attr_or_key(analysis, "department_codes", [])
        content_type = get_attr_or_key(analysis, "content_type")
        needs_clarification = bool(get_attr_or_key(result, "needs_clarification", False))

        sources = extract_sources(result)
        source_count = len(sources)

        sql_rows = extract_sql_rows(result)
        sql_row_count = len(sql_rows)
        sql_table = get_attr_or_key(get_attr_or_key(result, "sql_result"), "table_name")

        vector_status = extract_vector_status(result)
        vector_result_count = extract_vector_count(result)
        used_fallback = extract_vector_used_fallback(result)

        warnings = extract_warnings(result)

        issue_type = classify_issue_type(
            expected=expected,
            answer=answer,
            route=route,
            intent=intent,
            source_count=source_count,
            sql_row_count=sql_row_count,
            vector_status=vector_status,
            vector_result_count=vector_result_count,
            used_fallback=used_fallback,
            warnings=warnings,
        )

        output = {
            **base,
            "auto_eval": None,
            "issue_type": issue_type,
            "answer": answer,
            "answer_length": len(str(answer)),
            "source_count": source_count,
            "route": route,
            "intent": intent,
            "department_code": department_code,
            "department_codes": safe_json_dumps(department_codes),
            "content_type": content_type,
            "needs_clarification": needs_clarification,
            "sql_table": sql_table,
            "sql_row_count": sql_row_count,
            "vector_status": vector_status,
            "vector_result_count": vector_result_count,
            "used_fallback": used_fallback,
            "warnings": safe_json_dumps(warnings),
            "sources_json": safe_json_dumps(sources),
            "elapsed_sec": elapsed_sec,
            "error": "",
        }

        output["auto_eval"] = basic_evaluate(output)
        return output

    except Exception as exc:
        elapsed_sec = round(time.time() - started_at, 3)
        error_text = f"{type(exc).__name__}: {exc}"
        return {
            **base,
            "auto_eval": "FAIL_EXCEPTION",
            "issue_type": "EXCEPTION",
            "answer": "",
            "answer_length": 0,
            "source_count": 0,
            "route": None,
            "intent": None,
            "department_code": None,
            "department_codes": "[]",
            "content_type": None,
            "needs_clarification": False,
            "sql_table": None,
            "sql_row_count": 0,
            "vector_status": None,
            "vector_result_count": 0,
            "used_fallback": False,
            "warnings": "[]",
            "sources_json": "[]",
            "elapsed_sec": elapsed_sec,
            "error": error_text + "\n" + traceback.format_exc(),
        }

In [6]:
# ============================================================
# 5. 사전 라우팅 테스트
# 전체 평가 전 query_analyzer가 의도대로 작동하는지 빠르게 확인
# ============================================================

from src.rag.query_analyzer import QuestionAnalyzer

analyzer = QuestionAnalyzer()

routing_test_questions = [
    "AI미래학과는 어떤 인재를 양성하려고 해?",
    "AI컴퓨팅학과의 연락처가 문서에 나와 있어?",
    "AI대학 학과별 홈페이지 URL을 정리해줘.",
    "AI컴퓨팅학과와 AX학과를 비교해줘.",
    "나는 AI 서비스 적용에 관심이 있는데 어떤 학과가 맞아?",
    "AI컴퓨팅학과의 졸업 요건이 문서에 나와 있어?",
]

for question in routing_test_questions:
    a = analyzer.analyze(question)
    print("=" * 90)
    print("Q:", question)
    print("route:", a.route)
    print("intent:", a.intent)
    print("dept:", a.department_code)
    print("dept_codes:", a.department_codes)
    print("content_type:", a.content_type)
    print("sql_task:", a.sql_task_hint)
    print("missing:", a.missing_fields)

Q: AI미래학과는 어떤 인재를 양성하려고 해?
route: vector
intent: department_overview
dept: fx
dept_codes: ['fx']
content_type: None
sql_task: None
missing: []
Q: AI컴퓨팅학과의 연락처가 문서에 나와 있어?
route: sql
intent: office_contact_info
dept: aic
dept_codes: ['aic']
content_type: office_contact
sql_task: office_contact_lookup
missing: []
Q: AI대학 학과별 홈페이지 URL을 정리해줘.
route: sql
intent: department_homepage_info
dept: None
dept_codes: []
content_type: department_homepage
sql_task: department_homepage_lookup
missing: []
Q: AI컴퓨팅학과와 AX학과를 비교해줘.
route: vector
intent: comparison_info
dept: None
dept_codes: ['aic', 'ax']
content_type: None
sql_task: None
missing: []
Q: 나는 AI 서비스 적용에 관심이 있는데 어떤 학과가 맞아?
route: vector
intent: recommendation_info
dept: None
dept_codes: []
content_type: None
sql_task: None
missing: []
Q: AI컴퓨팅학과의 졸업 요건이 문서에 나와 있어?
route: sql
intent: requirement_info
dept: aic
dept_codes: ['aic']
content_type: requirement
sql_task: requirement_lookup
missing: []


In [7]:
# ============================================================
# 6. 전체 평가 실행
# ============================================================

results = []

for case in TEST_CASES:
    print(f"[{case['id']:03d}/100] {case['question']}")
    row = run_one_case(case)
    results.append(row)
    print(" ->", row["auto_eval"], "/", row["issue_type"], "/ sources:", row["source_count"], "/ route:", row["route"], "/ intent:", row["intent"])

results_df = pd.DataFrame(results)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
latest_csv_path = OUTPUT_DIR / "chatbot_test_results.csv"
timestamp_csv_path = OUTPUT_DIR / f"chatbot_test_results_{timestamp}.csv"

results_df.to_csv(latest_csv_path, index=False, encoding="utf-8-sig")
results_df.to_csv(timestamp_csv_path, index=False, encoding="utf-8-sig")

print("Saved latest:", latest_csv_path)
print("Saved timestamp:", timestamp_csv_path)

results_df.head()

[001/100] AI컴퓨팅학과는 어떤 학과야?
 -> PASS_ANSWER / OK / sources: 5 / route: vector / intent: department_overview
[002/100] AI시스템학과는 어떤 학과야?
 -> PASS_ANSWER / OK / sources: 4 / route: vector / intent: department_overview
[003/100] AX학과는 어떤 학과야?
 -> PASS_ANSWER / OK / sources: 5 / route: vector / intent: department_overview
[004/100] AI미래학과는 어떤 학과야?
 -> PASS_ANSWER / OK / sources: 5 / route: vector / intent: department_overview
[005/100] AI미래학과는 어떤 인재를 양성하려고 해?
 -> PASS_ANSWER / OK / sources: 5 / route: vector / intent: department_overview
[006/100] AI컴퓨팅학과의 교육목표를 알려줘.
 -> PARTIAL_NO_DIRECT_EVIDENCE / PARTIAL_NO_DIRECT_EVIDENCE / sources: 5 / route: vector / intent: department_overview
[007/100] AI시스템학과의 특징을 설명해줘.
 -> PASS_ANSWER / OK / sources: 4 / route: vector / intent: department_overview
[008/100] AX학과의 목표를 알려줘.
 -> PARTIAL_NO_DIRECT_EVIDENCE / PARTIAL_NO_DIRECT_EVIDENCE / sources: 5 / route: vector / intent: department_overview
[009/100] AI대학에는 어떤 학과들이 있어?
 -> PASS_ANSWER / OK / sources:

,id,category,question,expected,auto_eval,issue_type,answer,answer_length,source_count,route,...,needs_clarification,sql_table,sql_row_count,vector_status,vector_result_count,used_fallback,warnings,sources_json,elapsed_sec,error
0,1,department_overview,AI컴퓨팅학과는 어떤 학과야?,answer,PASS_ANSWER,OK,AI컴퓨팅학과는 인공지능과 컴퓨팅 관련 교육과 연구를 하는 학과입니다. 학부 및 대...,506,5,vector,...,False,NaN,0,searched,5,False,[],"[{""source_type"": ""vector"", ""title"": ""컴퓨팅 특강 <리...",8.933,
1,2,department_overview,AI시스템학과는 어떤 학과야?,answer,PASS_ANSWER,OK,AI시스템학과는 KAIST에서 인공지능 시스템의 미래를 설계하는 혁신의 중심 학과입...,421,4,vector,...,False,NaN,0,searched,5,False,[],"[{""source_type"": ""vector"", ""title"": ""AI시스템학과 소...",5.566,
2,3,department_overview,AX학과는 어떤 학과야?,answer,PASS_ANSWER,OK,"AX학과는 KAIST AI College 내 학과로, 다양한 전공선택 및 전공필수 ...",599,5,vector,...,False,NaN,0,searched,5,False,[],"[{""source_type"": ""vector"", ""title"": ""AX의 특강"", ...",5.032,
3,4,department_overview,AI미래학과는 어떤 학과야?,answer,PASS_ANSWER,OK,"AI미래학과는 AI 기술의 급격한 발전이 가져올 미래사회의 불확실성을 이해하고, 이...",858,5,vector,...,False,NaN,0,searched,5,False,[],"[{""source_type"": ""vector"", ""title"": ""D-2 학과설명회...",11.551,
4,5,department_overview,AI미래학과는 어떤 인재를 양성하려고 해?,answer,PASS_ANSWER,OK,AI미래학과는 다음과 같은 인재를 양성하려고 합니다.\n\n1. AI 전략가\n ...,751,5,vector,...,False,NaN,0,searched,5,False,[],"[{""source_type"": ""vector"", ""title"": ""D-2 학과설명회...",6.127,


In [8]:
# ============================================================
# 7. 평가 요약
# ============================================================

summary_eval = (
    results_df
    .groupby("auto_eval", dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

summary_issue = (
    results_df
    .groupby("issue_type", dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

summary_category = (
    results_df
    .groupby(["category", "auto_eval"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["category", "count"], ascending=[True, False])
)

summary_route_intent = (
    results_df
    .groupby(["route", "intent"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("[auto_eval 요약]")
display(summary_eval)

print("[issue_type 요약]")
display(summary_issue)

print("[category x auto_eval 요약]")
display(summary_category)

print("[route x intent 요약]")
display(summary_route_intent)

[auto_eval 요약]


,auto_eval,count
4,PASS_ANSWER,47
1,FAIL_DATA_MISSING_OR_RETRIEVAL,19
5,PASS_DATA_MISSING,15
6,PASS_REJECT,7
3,PARTIAL_NO_DIRECT_EVIDENCE,6
2,FAIL_SHOULD_REJECT,3
0,FAIL_DATA_MISSING_NOT_DETECTED,3


[issue_type 요약]


,issue_type,count
4,OK,47
1,DATA_MISSING_OR_RETRIEVAL_FAIL,19
2,EXPECTED_DATA_MISSING,15
3,EXPECTED_REJECTION,7
5,PARTIAL_NO_DIRECT_EVIDENCE,6
0,DATA_MISSING_NOT_DETECTED,3
6,REJECT_FAILED,3


[category x auto_eval 요약]


,category,auto_eval,count
0,admission,FAIL_DATA_MISSING_OR_RETRIEVAL,8
1,admission,PASS_ANSWER,2
2,comparison,PASS_ANSWER,10
3,contact,FAIL_DATA_MISSING_OR_RETRIEVAL,4
4,contact,PARTIAL_NO_DIRECT_EVIDENCE,1
5,contact_location,PASS_DATA_MISSING,2
6,course,FAIL_DATA_MISSING_OR_RETRIEVAL,6
7,course,PARTIAL_NO_DIRECT_EVIDENCE,2
8,course,PASS_DATA_MISSING,2
10,department_overview,PASS_ANSWER,8


[route x intent 요약]


,route,intent,count
12,vector,department_overview,12
11,vector,comparison_info,12
5,hybrid,admission_info,11
9,sql,person_info,11
10,sql,requirement_info,11
6,sql,course_info,10
14,vector,recommendation_info,10
8,sql,office_contact_info,7
13,vector,general_info,5
7,sql,department_homepage_info,4


In [9]:
# ============================================================
# 8. 실패/부분성공 케이스 확인
# ============================================================

problem_df = results_df[
    ~results_df["auto_eval"].astype(str).str.startswith("PASS")
].copy()

cols = [
    "id",
    "category",
    "question",
    "expected",
    "auto_eval",
    "issue_type",
    "route",
    "intent",
    "department_code",
    "department_codes",
    "source_count",
    "sql_row_count",
    "vector_status",
    "vector_result_count",
    "used_fallback",
    "warnings",
    "answer",
]

print("문제 케이스 수:", len(problem_df))
display(problem_df[cols])

문제 케이스 수: 31


,id,category,question,expected,auto_eval,issue_type,route,intent,department_code,department_codes,source_count,sql_row_count,vector_status,vector_result_count,used_fallback,warnings,answer
5,6,department_overview,AI컴퓨팅학과의 교육목표를 알려줘.,answer,PARTIAL_NO_DIRECT_EVIDENCE,PARTIAL_NO_DIRECT_EVIDENCE,vector,department_overview,aic,"[""aic""]",5,0,searched,5,False,[],제공된 자료에서는 AI컴퓨팅학과의 교육목표에 대한 구체적인 내용이 포함되어 있지 않...
7,8,department_overview,AX학과의 목표를 알려줘.,answer,PARTIAL_NO_DIRECT_EVIDENCE,PARTIAL_NO_DIRECT_EVIDENCE,vector,department_overview,ax,"[""ax""]",5,0,searched,5,False,[],제공된 자료에서는 AX학과의 구체적인 목표에 대한 직접적인 설명이나 명시된 내용이 ...
30,31,course,AI컴퓨팅학과의 교육과정을 알려줘.,answer,FAIL_DATA_MISSING_OR_RETRIEVAL,DATA_MISSING_OR_RETRIEVAL_FAIL,sql,course_info,aic,"[""aic""]",1,0,NaN,0,False,"[""OperationalError: (1045, \""Access denied for...",제공된 자료에서 질문에 대한 직접 근거를 찾을 수 없습니다. 데이터가 부족하거나 검...
32,33,course,AX학과의 교육과정을 알려줘.,answer,FAIL_DATA_MISSING_OR_RETRIEVAL,DATA_MISSING_OR_RETRIEVAL_FAIL,sql,course_info,ax,"[""ax""]",1,0,NaN,0,False,"[""OperationalError: (1045, \""Access denied for...",제공된 자료에서 질문에 대한 직접 근거를 찾을 수 없습니다. 데이터가 부족하거나 검...
33,34,course,AI미래학과의 교육과정을 알려줘.,answer,FAIL_DATA_MISSING_OR_RETRIEVAL,DATA_MISSING_OR_RETRIEVAL_FAIL,sql,course_info,fx,"[""fx""]",1,0,NaN,0,False,"[""OperationalError: (1045, \""Access denied for...",제공된 자료에서 질문에 대한 직접 근거를 찾을 수 없습니다. 데이터가 부족하거나 검...
34,35,course,AI대학에서 제공하는 교과목 정보를 정리해줘.,answer,PARTIAL_NO_DIRECT_EVIDENCE,PARTIAL_NO_DIRECT_EVIDENCE,sql,course_info,NaN,[],6,0,searched,5,False,"[""OperationalError: (1045, \""Access denied for...",제공된 자료에서 질문에 대한 직접 근거를 찾을 수 없습니다. 데이터가 부족하거나 검...
35,36,course,AI컴퓨팅학과에서 배울 수 있는 과목을 알려줘.,answer,FAIL_DATA_MISSING_OR_RETRIEVAL,DATA_MISSING_OR_RETRIEVAL_FAIL,sql,course_info,aic,"[""aic""]",1,0,NaN,0,False,"[""OperationalError: (1045, \""Access denied for...",제공된 자료에서 질문에 대한 직접 근거를 찾을 수 없습니다. 데이터가 부족하거나 검...
37,38,course,AX학과 과목 목록을 알려줘.,answer,FAIL_DATA_MISSING_OR_RETRIEVAL,DATA_MISSING_OR_RETRIEVAL_FAIL,sql,course_info,ax,"[""ax""]",1,0,NaN,0,False,"[""OperationalError: (1045, \""Access denied for...",제공된 자료에서 질문에 대한 직접 근거를 찾을 수 없습니다. 데이터가 부족하거나 검...
38,39,course,AI미래학과 과목 목록을 알려줘.,answer,FAIL_DATA_MISSING_OR_RETRIEVAL,DATA_MISSING_OR_RETRIEVAL_FAIL,sql,course_info,fx,"[""fx""]",1,0,NaN,0,False,"[""OperationalError: (1045, \""Access denied for...",제공된 자료에서 질문에 대한 직접 근거를 찾을 수 없습니다. 데이터가 부족하거나 검...
39,40,course,AI대학 학과별 주요 교과목을 비교해줘.,answer,PARTIAL_NO_DIRECT_EVIDENCE,PARTIAL_NO_DIRECT_EVIDENCE,vector,comparison_info,NaN,[],11,0,searched,12,False,[],AI대학 내 학과별 주요 교과목 정보는 제공된 자료에 구체적으로 나와 있지 않습니다...


In [10]:
# ============================================================
# 9. 데이터 부족으로 분류된 질문만 보기
# ============================================================

data_missing_df = results_df[
    results_df["issue_type"].isin([
        "EXPECTED_DATA_MISSING",
        "DATA_MISSING_NOT_DETECTED",
        "DATA_MISSING_OR_RETRIEVAL_FAIL",
        "PARTIAL_NO_DIRECT_EVIDENCE",
    ])
].copy()

display(data_missing_df[[
    "id",
    "category",
    "question",
    "expected",
    "auto_eval",
    "issue_type",
    "route",
    "intent",
    "source_count",
    "sql_row_count",
    "vector_result_count",
    "answer",
]])

,id,category,question,expected,auto_eval,issue_type,route,intent,source_count,sql_row_count,vector_result_count,answer
5,6,department_overview,AI컴퓨팅학과의 교육목표를 알려줘.,answer,PARTIAL_NO_DIRECT_EVIDENCE,PARTIAL_NO_DIRECT_EVIDENCE,vector,department_overview,5,0,5,제공된 자료에서는 AI컴퓨팅학과의 교육목표에 대한 구체적인 내용이 포함되어 있지 않...
7,8,department_overview,AX학과의 목표를 알려줘.,answer,PARTIAL_NO_DIRECT_EVIDENCE,PARTIAL_NO_DIRECT_EVIDENCE,vector,department_overview,5,0,5,제공된 자료에서는 AX학과의 구체적인 목표에 대한 직접적인 설명이나 명시된 내용이 ...
30,31,course,AI컴퓨팅학과의 교육과정을 알려줘.,answer,FAIL_DATA_MISSING_OR_RETRIEVAL,DATA_MISSING_OR_RETRIEVAL_FAIL,sql,course_info,1,0,0,제공된 자료에서 질문에 대한 직접 근거를 찾을 수 없습니다. 데이터가 부족하거나 검...
31,32,course,AI시스템학과의 교육과정을 알려줘.,data_missing,PASS_DATA_MISSING,EXPECTED_DATA_MISSING,sql,course_info,1,0,0,제공된 자료에서 AI시스템학과의 교육과정 또는 교과목에 대한 직접 근거를 찾을 수 ...
32,33,course,AX학과의 교육과정을 알려줘.,answer,FAIL_DATA_MISSING_OR_RETRIEVAL,DATA_MISSING_OR_RETRIEVAL_FAIL,sql,course_info,1,0,0,제공된 자료에서 질문에 대한 직접 근거를 찾을 수 없습니다. 데이터가 부족하거나 검...
33,34,course,AI미래학과의 교육과정을 알려줘.,answer,FAIL_DATA_MISSING_OR_RETRIEVAL,DATA_MISSING_OR_RETRIEVAL_FAIL,sql,course_info,1,0,0,제공된 자료에서 질문에 대한 직접 근거를 찾을 수 없습니다. 데이터가 부족하거나 검...
34,35,course,AI대학에서 제공하는 교과목 정보를 정리해줘.,answer,PARTIAL_NO_DIRECT_EVIDENCE,PARTIAL_NO_DIRECT_EVIDENCE,sql,course_info,6,0,5,제공된 자료에서 질문에 대한 직접 근거를 찾을 수 없습니다. 데이터가 부족하거나 검...
35,36,course,AI컴퓨팅학과에서 배울 수 있는 과목을 알려줘.,answer,FAIL_DATA_MISSING_OR_RETRIEVAL,DATA_MISSING_OR_RETRIEVAL_FAIL,sql,course_info,1,0,0,제공된 자료에서 질문에 대한 직접 근거를 찾을 수 없습니다. 데이터가 부족하거나 검...
36,37,course,AI시스템학과에서 배울 수 있는 과목을 알려줘.,data_missing,PASS_DATA_MISSING,EXPECTED_DATA_MISSING,sql,course_info,1,0,0,제공된 자료에서 AI시스템학과의 교육과정 또는 교과목에 대한 직접 근거를 찾을 수 ...
37,38,course,AX학과 과목 목록을 알려줘.,answer,FAIL_DATA_MISSING_OR_RETRIEVAL,DATA_MISSING_OR_RETRIEVAL_FAIL,sql,course_info,1,0,0,제공된 자료에서 질문에 대한 직접 근거를 찾을 수 없습니다. 데이터가 부족하거나 검...


In [11]:
# ============================================================
# 10. source_count=0인데 expected=answer인 위험 케이스
# ============================================================

no_source_answer_df = results_df[
    (results_df["expected"] == "answer")
    & (results_df["source_count"] == 0)
].copy()

display(no_source_answer_df[[
    "id",
    "category",
    "question",
    "auto_eval",
    "issue_type",
    "route",
    "intent",
    "source_count",
    "sql_row_count",
    "vector_status",
    "vector_result_count",
    "answer",
]])

,id,category,question,auto_eval,issue_type,route,intent,source_count,sql_row_count,vector_status,vector_result_count,answer


In [12]:
# ============================================================
# 11. 라우팅 핵심 질문만 재확인
# ============================================================

key_ids = [5, 11, 21, 32, 51, 73, 80, 88, 91, 99]

key_df = results_df[results_df["id"].isin(key_ids)].copy()
display(key_df[[
    "id",
    "question",
    "expected",
    "auto_eval",
    "issue_type",
    "route",
    "intent",
    "department_code",
    "department_codes",
    "source_count",
    "sql_row_count",
    "vector_status",
    "vector_result_count",
    "warnings",
    "answer",
]])

,id,question,expected,auto_eval,issue_type,route,intent,department_code,department_codes,source_count,sql_row_count,vector_status,vector_result_count,warnings,answer
4,5,AI미래학과는 어떤 인재를 양성하려고 해?,answer,PASS_ANSWER,OK,vector,department_overview,fx,"[""fx""]",5,0,searched,5,[],AI미래학과는 다음과 같은 인재를 양성하려고 합니다.\n\n1. AI 전략가\n ...
10,11,AI컴퓨팅학과와 AX학과를 비교해줘.,answer,PASS_ANSWER,OK,vector,comparison_info,NaN,"[""aic"", ""ax""]",6,0,searched,6,[],AI컴퓨팅학과와 AX학과의 비교는 다음과 같습니다.\n\n1. AI컴퓨팅학과 (ai...
20,21,나는 AI 서비스 적용에 관심이 있는데 어떤 학과가 맞아?,answer,PASS_ANSWER,OK,vector,recommendation_info,NaN,[],8,0,searched,8,[],AI 서비스 적용에 관심이 있다면 다음 학과들이 적합할 수 있습니다:\n\n1. A...
31,32,AI시스템학과의 교육과정을 알려줘.,data_missing,PASS_DATA_MISSING,EXPECTED_DATA_MISSING,sql,course_info,ai_systems,"[""ai_systems""]",1,0,NaN,0,"[""OperationalError: (1045, \""Access denied for...",제공된 자료에서 AI시스템학과의 교육과정 또는 교과목에 대한 직접 근거를 찾을 수 ...
50,51,AI컴퓨팅학과의 졸업 요건이 문서에 나와 있어?,data_missing,PASS_DATA_MISSING,EXPECTED_DATA_MISSING,sql,requirement_info,aic,"[""aic""]",1,0,NaN,0,"[""OperationalError: (1045, \""Access denied for...",제공된 자료에서 AI컴퓨팅학과의 졸업/수료/이수/논문 요건에 대한 직접 근거를 찾을...
72,73,AI컴퓨팅학과의 연락처가 문서에 나와 있어?,answer,FAIL_DATA_MISSING_OR_RETRIEVAL,DATA_MISSING_OR_RETRIEVAL_FAIL,sql,office_contact_info,aic,"[""aic""]",1,0,NaN,0,"[""OperationalError: (1045, \""Access denied for...",제공된 자료에서 해당 학과의 연락처에 대한 직접 근거를 찾을 수 없습니다. 학과 사...
79,80,AI대학 학과별 홈페이지 URL을 정리해줘.,answer,PASS_ANSWER,OK,sql,department_homepage_info,NaN,[],1,0,NaN,0,"[""OperationalError: (1045, \""Access denied for...",제공된 자료에서 학과별 대표 홈페이지 URL 정보를 찾을 수 없습니다. `depar...
87,88,AI대학 졸업 요건을 문서 근거와 함께 설명해줘.,data_missing,PASS_DATA_MISSING,EXPECTED_DATA_MISSING,sql,requirement_info,NaN,[],1,0,NaN,0,"[""OperationalError: (1045, \""Access denied for...",제공된 자료에서 AI대학 학과별 졸업/수료/이수/논문 요건에 대한 직접 근거를 찾을...
90,91,서울대학교 AI대학원 입학 정보를 알려줘.,reject,PASS_REJECT,EXPECTED_REJECTION,clarify,admission_info,NaN,[],0,0,NaN,0,[],"현재 수집된 데이터는 KAIST AI College 관련 4개 학과(AI컴퓨팅학과,..."
98,99,서울대학교 AI대학원과 KAIST AI대학을 교수진 기준으로 비교해줘.,reject,PASS_REJECT,EXPECTED_REJECTION,clarify,comparison_info,NaN,[],0,0,NaN,0,[],"현재 수집된 데이터는 KAIST AI College 관련 4개 학과(AI컴퓨팅학과,..."


In [13]:
# ============================================================
# 12. 결과 파일 위치 출력
# ============================================================

print("latest_csv_path:", latest_csv_path)
print("timestamp_csv_path:", timestamp_csv_path)
print("\n다음 단계:")
print("1. FAIL_NO_SOURCE / FAIL_UNEXPECTED_CLARIFY는 RAG 구조 문제 가능성이 큽니다.")
print("2. PASS_DATA_MISSING은 정상입니다. 실제 데이터 부족을 안전하게 말한 것입니다.")
print("3. FAIL_DATA_MISSING_NOT_DETECTED는 데이터 부족인데 모델이 답한 위험 케이스입니다.")
print("4. PARTIAL_FALLBACK은 답변은 나왔지만 fallback 근거가 섞인 케이스입니다.")
print("5. source_count=0인데 PASS_ANSWER가 나오면 평가 기준을 더 강화해야 합니다.")

latest_csv_path: C:\Users\Playdata\workspace\SKN28-third-2TEAM\notebooks\chatbot_eval_outputs\chatbot_test_results.csv
timestamp_csv_path: C:\Users\Playdata\workspace\SKN28-third-2TEAM\notebooks\chatbot_eval_outputs\chatbot_test_results_20260602_173854.csv

다음 단계:
1. FAIL_NO_SOURCE / FAIL_UNEXPECTED_CLARIFY는 RAG 구조 문제 가능성이 큽니다.
2. PASS_DATA_MISSING은 정상입니다. 실제 데이터 부족을 안전하게 말한 것입니다.
3. FAIL_DATA_MISSING_NOT_DETECTED는 데이터 부족인데 모델이 답한 위험 케이스입니다.
4. PARTIAL_FALLBACK은 답변은 나왔지만 fallback 근거가 섞인 케이스입니다.
5. source_count=0인데 PASS_ANSWER가 나오면 평가 기준을 더 강화해야 합니다.
